# 1). Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.cluster import KMeans 
from sklearn.metrics import silhouette_score
from scipy.stats.mstats import trimmed_var
from sklearn.decomposition import PCA

# 2). Load Dataset

In [2]:
df=pd.read_csv("/workspaces/Customer_Segmentation_KMeans_Clustering/data/clean_customer_data.csv")
df.head()

,CustomerID,TotalSpent,Orders,ItemsPurchased,CountriesPurchased,AvgOrderValue,Recency
0,12347.0,4310.00,7,2458,1,615.714286,1
1,12348.0,1797.24,4,2341,1,449.310000,74
2,12349.0,1757.55,1,631,1,1757.550000,18
3,12350.0,334.40,1,197,1,334.400000,309
4,12352.0,1240.73,4,380,1,310.182500,35


# 3). Explore

## a) Total Spent

In [3]:
df.TotalSpent.describe()

count      4239.00000
mean       1682.84127
std        6973.03743
min           0.85000
25%         294.18000
50%         623.94000
75%        1471.44500
max      259657.30000
Name: TotalSpent, dtype: float64

In [4]:
# Explore Total Spent
fig=px.histogram(df, x="TotalSpent",
                 title="Distribution of Total monetary amount spent on all orders")
fig.update_layout(xaxis_title='Total amount', yaxis_title='Count')
fig.show()



> The distribution of TotalSpent suggests that customer spending is highly positively (right) skewed, with a small number of customers spending substantially more than the majority.

#### Key Insights
- The median spending is 623.94, meaning that half of the customers spent less than approximately 624 monetary units during the observation period.
- The mean spending is 1,682.84, which is almost three times larger than the median. This large difference indicates that a relatively small number of high-spending customers are pulling the average upward.
- The maximum spending of 259,657.30 is extremely high compared to the 75th percentile (1,471.45), suggesting the presence of significant outliers, likely representing wholesale or very high-value customers.
- The interquartile range (IQR) extends from 294.18 to 1,471.45, indicating that the middle 50% of customers spent within this range. Most customers therefore have relatively modest spending compared to the few extreme spenders.
- The large standard deviation (6,973.04), which is much greater than the mean, further confirms substantial variability in customer spending.

This distribution is important because K-Means is distance-based and sensitive to extreme values. Customers with exceptionally high spending can disproportionately influence the cluster centroids, potentially resulting in clusters that reflect spending extremes rather than meaningful customer segments.

Before fitting the K-Means model, it would be important to Scale the features using StandardScaler.

> Overall, the distribution suggests a typical retail purchasing pattern, where most customers spend relatively small amounts, while a small proportion of customers contribute disproportionately large amounts of revenue. This is a common characteristic of customer transaction data and reinforces the value of segmentation to identify and treat these customer groups differently.

## b) Orders

In [5]:

px.bar(df["Orders"], title="Distribution of total orders placed by customers")
fig.update_layout(xaxis_title='Total orders ', yaxis_title='Customers')
fig.show()

In [6]:
df["Orders"].value_counts()

Orders
1      1523
2       862
3       500
4       378
5       227
6       160
7       133
8        80
9        62
10       48
11       44
12       37
15       21
13       21
17       20
14       17
16       16
18       10
20        9
19        6
24        5
21        5
29        5
22        5
28        4
23        4
26        4
40        2
44        2
68        2
46        2
27        2
34        2
39        2
31        1
182       1
83        1
91        1
37        1
35        1
54        1
36        1
33        1
56        1
51        1
53        1
50        1
32        1
45        1
41        1
25        1
30        1
60        1
Name: count, dtype: int64

In [7]:
df["Orders"].describe()

count    4239.000000
mean        3.820712
std         6.003651
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max       182.000000
Name: Orders, dtype: float64

The Orders feature indicates that most customers placed relatively few orders during the observation period. The median number of orders is 2, meaning that half of all customers placed two or fewer orders. Similarly, the 75th percentile is 4, indicating that three-quarters of customers made no more than four purchases. However, the mean number of orders (3.82) is noticeably higher than the median due to a small number of customers with exceptionally high purchase frequencies. The most active customer placed 182 orders, far exceeding the typical customer and suggesting the presence of highly loyal or business customers. The standard deviation (6.00) also reflects considerable variation in purchasing frequency across the customer base.

> This distribution suggests that the retailer's customer base is largely composed of occasional or infrequent buyers, while a relatively small segment consists of repeat or highly loyal customers. 

## c) Items Purchased

In [8]:
df["ItemsPurchased"].describe()

count      4239.000000
mean        981.146497
std        3702.512354
min           1.000000
25%         152.000000
50%         348.000000
75%         876.000000
max      153567.000000
Name: ItemsPurchased, dtype: float64

In [9]:

fig = px.histogram(
    df,
    x="ItemsPurchased",
    nbins=50,
    title="Distribution of Items Purchased per Customer"
)

fig.update_layout(
    xaxis_title="Items Purchased",
    yaxis_title="Number of Customers"
)

fig.show()

In [10]:
df["ItemsPurchased"].describe()

count      4239.000000
mean        981.146497
std        3702.512354
min           1.000000
25%         152.000000
50%         348.000000
75%         876.000000
max      153567.000000
Name: ItemsPurchased, dtype: float64

> The ItemsPurchased feature exhibits a highly right-skewed distribution, indicating that while most customers purchase relatively small quantities of products, a small number of customers buy exceptionally large volumes. 
- The median customer purchased 348 units, meaning that half of all customers bought fewer than 348 units during the observation period. In contrast, the mean is 981 units, which is almost three times the median, suggesting that a small group of high-volume purchasers is pulling the average upward. 
- The maximum purchase volume of 153,567 units is substantially larger than the 75th percentile of 876 units, highlighting the presence of extreme outliers. These customers are likely to represent wholesale or bulk purchasers rather than typical retail shoppers. The large standard deviation (3,702.51) further reflects the considerable variability in purchasing volumes across customers.

#### Business insight

> This distribution suggests that the retailer serves multiple customer types. While the majority of customers make relatively modest purchases, a small segment purchases products in very large quantities, potentially representing business or wholesale clients. Identifying these high-volume customers is one of the objectives of customer segmentation, as they may require different pricing, inventory allocation, or marketing strategies.
 

## d) Countries Purchased

In [11]:
df["CountriesPurchased"].value_counts()

CountriesPurchased
1    4232
2       7
Name: count, dtype: int64


> The CountriesPurchased feature shows very little variation across customers. Out of 4,239 customers, 4,232 (99.8%) made purchases from only one country, while only 7 customers (0.2%) purchased from two countries. This indicates that nearly all customers conducted their transactions within a single country during the observation period. As a result, this feature provides minimal discriminatory power for customer segmentation and is unlikely to contribute meaningfully to the K-Means clustering process.

#### Business insight
The overwhelming majority of customers appear to purchase from a single country, suggesting that cross-country purchasing is extremely rare in this dataset. This may reflect the retailer's customer base or shipping operations, where customers typically transact within their country of residence.

#### From a Modeling perspective:
Since CountriesPurchased is nearly constant across all customers, it contributes little to distinguishing customer groups. Features with very low variance generally provide limited information to distance-based algorithms such as K-Means and may be considered for removal before model training. Excluding this feature can simplify the model without sacrificing meaningful information.

### e) Average Order Value

In [12]:
df["AvgOrderValue"].describe()

count     4239.000000
mean       369.296741
std        491.547989
min          0.850000
25%        173.098333
50%        283.180000
75%        418.181667
max      19809.750000
Name: AvgOrderValue, dtype: float64

In [13]:
fig=px.histogram(df["AvgOrderValue"], title="Average monetary value of each order")
fig.update_layout(xaxis_title="Average Order Value", yaxis_title="Count")
fig.show()

The AvgOrderValue feature exhibits a positively skewed distribution, indicating that while most customers place orders of relatively modest value, a small number make exceptionally high-value purchases. The median average order value is 283.18, meaning that half of the customers spend less than this amount per order. The mean average order value (369.30) exceeds the median, suggesting that a few high-value customers are increasing the overall average. Furthermore, the maximum average order value of 19,809.75 is substantially higher than the 75th percentile (418.18), highlighting the presence of extreme outliers. The large standard deviation (491.55) also reflects considerable variation in customers' average spending per order.

> Business insight

The distribution suggests that most customers make relatively low to moderate-value purchases, whereas a small segment consistently places high-value orders. These customers may represent premium buyers or wholesale clients whose purchasing behavior differs significantly from the typical customer. 

> Modeling implication

The pronounced right skew and presence of high-value outliers indicate that AvgOrderValue may disproportionately influence the distance calculations used by K-Means clustering. Therefore, the feature should be standardized before clustering. Depending on the severity of the skewness observed in visualizations, a log transformation may also be considered to reduce the influence of extreme values while preserving the relative differences between customers.

## d) Recency

In [14]:
df["Recency"].describe()

count    4239.000000
mean       92.907997
std       100.647266
min         0.000000
25%        17.000000
50%        51.000000
75%       144.500000
max       373.000000
Name: Recency, dtype: float64

In [15]:
fig=px.histogram(df["Recency"], title="Distribution of recency")
fig.update_layout(xaxis_title="Recency")
fig.show()

> The Recency feature measures the number of days since a customer's last purchase and shows considerable variation in customer activity levels.
- The median recency is 51 days, indicating that half of the customers made their most recent purchase within the last 51 days of the observation period. The mean recency (92.91 days) is substantially higher than the median, suggesting a moderately right-skewed distribution where a smaller group of customers has been inactive for extended periods. The recency values range from 0 days, representing customers who made a purchase on the final day of the observation period, to 373 days, indicating customers who had not made a purchase for over a year. The relatively large standard deviation (100.65 days) further highlights the wide variation in customer activity.

> Business insight

The distribution indicates that the retailer has a mix of active and inactive customers. Customers with low recency values are recent purchasers and are more likely to be engaged with the business, while those with high recency values may represent dormant or at-risk customers who have not purchased for a long period. These differences in purchasing recency are valuable for identifying customer segments that may require different retention or re-engagement strategies.

## f) Correlation Analysis

In [16]:
corr=df.drop(columns=["CustomerID", "CountriesPurchased"]).corr()
corr.style.background_gradient(axis=None)

,TotalSpent,Orders,ItemsPurchased,AvgOrderValue,Recency
TotalSpent,1.000000,0.558532,0.889305,0.390556,-0.129485
Orders,0.558532,1.000000,0.587912,0.092145,-0.285282
ItemsPurchased,0.889305,0.587912,1.000000,0.309241,-0.143040
AvgOrderValue,0.390556,0.092145,0.309241,1.000000,-0.049981
Recency,-0.129485,-0.285282,-0.143040,-0.049981,1.000000


The correlation analysis reveals several meaningful relationships between the engineered customer features while indicating that each feature captures a distinct aspect of customer purchasing behaviour.

- The strongest positive correlation is observed between TotalSpent and ItemsPurchased (r = 0.889), indicating that customers who purchase larger quantities of products generally spend more overall. This relationship is expected because total spending is directly influenced by the quantity of items purchased.

- A moderate positive correlation exists between TotalSpent and Orders (r = 0.559), suggesting that customers who place more orders tend to spend more. However, the correlation is not perfect, implying that the value of each order also plays an important role in determining total spending.

- Similarly, ItemsPurchased and Orders exhibit a moderate positive correlation (r = 0.588), indicating that customers with more purchase occasions generally buy more units. Nevertheless, differences in order size mean that purchasing frequency alone does not fully explain purchasing volume.

- The AvgOrderValue feature shows relatively weak to moderate positive correlations with the other spending-related variables, particularly TotalSpent (r = 0.391). This suggests that customers with higher average order values tend to spend more overall, although average order value represents a distinct purchasing behaviour that is not solely determined by purchase frequency or purchasing volume.

- The Recency feature displays weak negative correlations with all purchasing-related features. The strongest of these is with Orders (r = -0.285), indicating that customers who place more orders generally tend to have made more recent purchases (lower recency values). Similarly, weak negative correlations with TotalSpent and ItemsPurchased suggest that customers who spend more and purchase larger quantities are, on average, slightly more active. However, these relationships are relatively weak, implying that recency provides additional information beyond spending and purchasing behaviour.

> Overall, the correlation analysis suggests that although some features are related, none are perfectly correlated. Each feature contributes unique information about customer behaviour, making them collectively suitable for customer segmentation using K-Means clustering.

# 4) Modelling

## Split

In [17]:
# Create copies of the dataframe to work with
df_original = df.copy()
df_model=df.copy()
df_copy=df.copy()
# Set features
X=df_copy.drop(columns=["CustomerID", "CountriesPurchased"])
print("X:", X.shape)

X: (4239, 5)


In [18]:
X.head()

,TotalSpent,Orders,ItemsPurchased,AvgOrderValue,Recency
0,4310.00,7,2458,615.714286,1
1,1797.24,4,2341,449.310000,74
2,1757.55,1,631,1757.550000,18
3,334.40,1,197,334.400000,309
4,1240.73,4,380,310.182500,35


## Build Models

### Model 1 (Raw Model No Transformation)

In [19]:
# Set parameters
k_clusters=range(2, 13, 1)
inertia_errors_1=[]
silhouette_scores_1=[]

scaler = StandardScaler()
X_scaled_1 = scaler.fit_transform(X)
for k in k_clusters:
    # Build model
    model_1=make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    
    )
    # Fit model
    model_1.fit(X)
    # Calculate inertia_errors and append to list
    inertia_errors_1.append(model_1.named_steps['kmeans'].inertia_)
    # Calculate silhouette_scores and append to list
    silhouette_scores_1.append(silhouette_score(X_scaled_1, model_1.named_steps['kmeans'].labels_))

print("Inertia errors:", inertia_errors_1)
print("Silhouette Scores:", silhouette_scores_1)


Inertia errors: [15193.00044884616, 11629.539838867415, 10186.001383210329, 7656.345656971492, 6357.185312762176, 5087.460052878807, 4544.163556918152, 4187.342821052463, 3776.4879431454706, 3349.2453035670237, 2915.709396247774]
Silhouette Scores: [0.894852918768882, 0.472879164007113, 0.4987732526397285, 0.5003073402784707, 0.5076490627290512, 0.5112462705523716, 0.5095695088465305, 0.44209281622116653, 0.38995036520705373, 0.38995295300886823, 0.40489784272127866]


In [20]:
fig=px.line(x=k_clusters, y=inertia_errors_1, title="Line plot of Inertia errors")
fig.update_layout(xaxis_title='k_clusters', yaxis_title='Inertia errors')
fig.show()

The inertia errors behave as expected:They decrease monotonically as k increases. The elbow plot will help identify where adding more clusters starts yielding diminishing returns.

In [21]:
fig=px.line(x=k_clusters, y=silhouette_scores_1, title="Line plot of Silouette scores")
fig.update_layout(xaxis_title='k_clusters', yaxis_title='silhouette scores')
fig.show()

### Model 2 (k=2)

### Investigate model cluster k=2
> Model with k=2 clusters registers unusually high silhoutte scores compared to the rest. We need to investigate it

In [22]:


model_k_2 = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=2, random_state=42)
)

model_k_2.fit(X)

labels = model_k_2.named_steps["kmeans"].labels_


print(pd.Series(labels).value_counts())

1    4211
0      28
Name: count, dtype: int64


### Cluster 2 interpretation
Out of 4,239 customers, one cluster contains 99.3% of all customers, while the other contains only 0.7%. This is a classic sign that K-Means is isolating a handful of extreme customers (outliers) from the rest of the population.

> Why is the silhouette score so high compared to the rest?
- The silhouette score measures how well separated the clusters are. Those 28 customers are probably:exceptionally high spenders, exceptionally high-volume purchasers, customers with many orders, or a combination of these.
- Because they are so far away from the majority of customers in the feature space, K-Means separates them very cleanly, resulting in a very high silhouette score of 0.895.
- So the high score is not necessarily telling us that the two customer segments are the best business solution. Instead, it's telling us that there is a small group of customers who are very different from everyone else.

Should we choose k = 2?, No, For two reasons:
> 1 Business usefulness
A segmentation where one cluster contains 4,211 customers and the other contains 28 customers provides very limited insight. It essentially says: (Everyone else, Extreme customers)
While that's informative, it doesn't help us understand the diversity within the 4,211 customers.

> 2 Our objective
The goal of customer segmentation is usually to identify several meaningful customer profiles, not just separate outliers from the majority.

> Therefore we need to investigate further, the clusters with modest silhouette scores (5, 6, 7)

## Plots Interpretation
> Looking at the elbow plot together with the silhouette scores:
- The inertia drops rapidly from k = 2 → 5 then continues to decrease from 5 → 7, after about k = 7, the curve starts to flatten considerably.
- It is also clear that the elbow is around k = 7.
- Also at k=7, the silouette score is decent so lets build a model with k=7 then investigate

### Model 3 (k=7)

In [23]:
## Model k=7
model_k_7=make_pipeline(StandardScaler(),
                        KMeans(n_clusters=7, random_state=42))
model_k_7.fit(X)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('kmeans', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['TotalSpent','Orders','ItemsPurchased','AvgOrderValue','Recency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [24]:
# Evaluate Cluster Sizes
# Assign Customers to clusters
df_copy["Cluster"] = model_k_7.named_steps["kmeans"].labels_

In [25]:
# Size of each cluster
df_copy["Cluster"].value_counts().sort_index()

Cluster
0      94
1    2767
2      27
3    1029
4       3
5     318
6       1
Name: count, dtype: int64

#### These Cluster sizes are not ideal
Out of 4,239 customers, one cluster contains just a single customer, and another contains only three customers.
> What does this mean?
K-Means is likely treating a few extreme customers as their own clusters instead of grouping them with similar customers. This often happens when there are outliers in the data (just like we had seen in our data exploration).
- Based on these cluster sizes, we need to reconsider our choices.
- A good customer segmentation should produce clusters that are reasonably distinct, interpretable, and large enough to be useful.
- Clusters containing 1 or 3 customers are difficult to analyze and are generally not actionable for business decisions.
> Let's do a comparison with cluster sizes of clusters 5 and 6

### Model 4 (k=[5,6,7])

### Compare with other cluster sizes (5, 6, 7)

In [26]:
for k in [5, 6, 7]:
    model_k_5_6_7 = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )

    model_k_5_6_7.fit(X)

    labels = model_k_5_6_7.named_steps["kmeans"].labels_

    print(f"\n{k} Clusters")
    print(pd.Series(labels).value_counts().sort_index())


5 Clusters
0      73
1    3065
2      28
3    1070
4       3
Name: count, dtype: int64

6 Clusters
0      59
1    2784
2      27
3    1033
4       3
5     333
Name: count, dtype: int64

7 Clusters
0      94
1    2767
2      27
3    1029
4       3
5     318
6       1
Name: count, dtype: int64


#### These findings actually reinforce what we observed during EDA: 
> The data contains a small number of extremely unusual customers that K-Means consistently isolates into their own clusters.
- Similar patterns are observed across clusters 5, 6,7 
- Based on these findings, it is likely that the highly skewed distributions and extreme values identified during the exploratory data analysis are influencing the K-Means algorithm.

> To reduce the impact of these extreme observations while preserving all customers in the analysis, a logarithmic transformation will be applied to the skewed numerical features before re-scaling the data and rebuilding the clustering model. 

### Model 5 (Model on Log Transformed data)

#### Perform log transformation then remodel

In [27]:
df_log=df.copy()

df_log[["TotalSpent", "Orders", "ItemsPurchased", "AvgOrderValue"]] = np.log1p(
    df_log[["TotalSpent", "Orders", "ItemsPurchased", "AvgOrderValue"]]
)

In [28]:
# Rebuild model
# Set features
X2=df_log.drop(columns=['CountriesPurchased', 'CustomerID'])
print("X2:", X2.shape)
X2.head()

X2: (4239, 5)


,TotalSpent,Orders,ItemsPurchased,AvgOrderValue,Recency
0,8.368925,2.079442,7.807510,6.424406,1
1,7.494564,1.609438,7.758761,6.109936,74
2,7.472245,0.693147,6.448889,7.472245,18
3,5.815324,0.693147,5.288267,5.815324,309
4,7.124261,1.609438,5.942799,5.740380,35


In [29]:
# Set parameters
k_clusters=range(2, 13, 1)
inertia_errors_2=[]
silhouette_scores_2=[]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X2)
for k in k_clusters:
    # Build model
    model_log=make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    
    )
    # Fit model
    model_log.fit(X2)
    # Calculate inertia_errors and append to list
    inertia_errors_2.append(model_log.named_steps['kmeans'].inertia_)
    # Calculate silhouette_scores and append to list
    silhouette_scores_2.append(silhouette_score(X_scaled, model_log.named_steps['kmeans'].labels_))

print("Inertia errors:", inertia_errors_2)
print("Silhouette Scores:", silhouette_scores_2)


Inertia errors: [12441.311142232835, 9667.30787881071, 8022.498563113711, 6960.764476487521, 6019.632657220023, 5425.459449875089, 5016.635244770998, 4550.23918180756, 4325.542320434912, 4040.548498541376, 3844.6034929530083]
Silhouette Scores: [0.35144329449821116, 0.2922633865903292, 0.2791920783196309, 0.28189581933838653, 0.2760087715571978, 0.273104005984237, 0.2589971081504425, 0.265560169294745, 0.2511407667820957, 0.24905618992756656, 0.24443597463770558]


In [30]:
fig=px.line(x=k_clusters, y=inertia_errors_2, title="Line plot of Inertia errors")
fig.update_layout(xaxis_title='k_clusters', yaxis_title='Inertia errors')
fig.show()

In [32]:
fig=px.line(x=k_clusters, y=silhouette_scores_2, title="Line plot of Silhouette scores")
fig.update_layout(xaxis_title='k_clusters', yaxis_title='silhouette scores')

fig.show()


#### First observation: The silhouette scores decreased
- Before the log transformation: Best meaningful silhouette ≈ 0.511 (k = 7)
- After the log transformation: Best silhouette = 0.351 (k = 2)
At first glance, this looks like the model got worse. However, as we know, a lower silhouette score does not necessarily mean a worse clustering solution.
> Why did the silhouette scores decrease?:
- The log transformation compressed the large values of: TotalSpent, Orders, ItemsPurchased, AvgOrderValue
- Previously, the extreme customers were very far away from the rest of the data, making them easy to separate. That large separation inflated the silhouette score.
- After the transformation: the extreme customers moved closer to the rest, clusters became less dramatically separated, so the silhouette score naturally decreased.

### Compare Cluster sizes again

In [36]:
for k in range(2,8):
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )

    model.fit(X2)

    labels = model.named_steps["kmeans"].labels_

    print(f"\n{k} Clusters")
    print(pd.Series(labels).value_counts().sort_index())


2 Clusters
0    1924
1    2315
Name: count, dtype: int64

3 Clusters
0    2012
1    1180
2    1047
Name: count, dtype: int64

4 Clusters
0    1534
1     806
2     877
3    1022
Name: count, dtype: int64

5 Clusters
0     659
1     722
2     827
3    1441
4     590
Name: count, dtype: int64

6 Clusters
0     684
1     413
2     481
3     917
4    1114
5     630
Name: count, dtype: int64

7 Clusters
0    409
1    336
2    463
3    776
4    690
5    574
6    991
Name: count, dtype: int64


> The cluster size distribution after the log transformation is considerably more balanced, indicating that the influence of extreme customer behaviour has been substantially reduced.
- For 2, 3, and 4 clusters, customers are distributed relatively evenly across the groups, suggesting that the transformation successfully compressed the range of highly skewed variables and allowed K-Means to identify broader patterns in purchasing behaviour.
- For the 5, 6, and 7 cluster solutions, most clusters remain well balanced, with the exception of one smaller cluster containing between 50 and 62 customers. 
- Unlike the previous model, where some clusters contained only one or three customers, these smaller clusters are now large enough to represent a meaningful niche customer segment rather than isolated outliers. 
- This indicates that the log transformation preserved the presence of unusual purchasing behaviours while preventing individual extreme customers from dominating the clustering process.

- Overall, the transformed data produced clusters that are more representative of the underlying customer population and therefore offer greater business value for customer segmentation.

### Selecting final model

- The final K-Means model will be built using 5 clusters.
- Although the highest silhouette score was obtained with 2 clusters, that solution produces overly broad customer groups that offer limited business insight. 
- After applying a logarithmic transformation to reduce the influence of highly skewed features, the Elbow Method indicates that five clusters represents an appropriate balance between model complexity and within-cluster variation.- Furthermore, the five-cluster solution produces well-balanced customer segments
- Increasing the number of clusters beyond five results in lower silhouette scores and increasingly fragmented customer groups without substantial improvements in clustering quality.
- Therefore, the five-cluster solution was is the optimal balance between statistical performance, cluster stability, and business interpretability.

### Final Model (Log Transformed, k=5)

In [37]:
# Final model
final_model=make_pipeline(StandardScaler(),
                        KMeans(n_clusters=5, random_state=42))
final_model.fit(X2)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('kmeans', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['TotalSpent','Orders','ItemsPurchased','AvgOrderValue','Recency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


### Assign Customers to Clusters and Save to CSV

In [50]:
# Assign Customers to clusters
df_model["Cluster"] = final_model.named_steps["kmeans"].labels_
df_model.to_csv("clusters.csv", index=False)

In [40]:
df_model.head()

,CustomerID,TotalSpent,Orders,ItemsPurchased,CountriesPurchased,AvgOrderValue,Recency,Cluster
0,12347.0,4310.00,7,2458,1,615.714286,1,2
1,12348.0,1797.24,4,2341,1,449.310000,74,0
2,12349.0,1757.55,1,631,1,1757.550000,18,0
3,12350.0,334.40,1,197,1,334.400000,309,1
4,12352.0,1240.73,4,380,1,310.182500,35,3
